# Generate 2048 parameter sets to effectively sample the parameter space of WOMBAT-mid run in RYF of ACCESS-OM2

### imports

In [1]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy as sci
from scipy.stats import qmc

wrkdir = "/g/data/vn19/pjb581/SOTS-Optimization/WOMBATfull/data"

# print versions of packages
print("python version =",sys.version[:5])
print("numpy version =", np.__version__)
print("pandas version =", pd.__version__)
print("scipy version =", sci.__version__)
print("matplotlib version =", sys.modules[plt.__package__].__version__)

os.chdir(wrkdir)
os.listdir()


python version = 3.10.
numpy version = 1.26.4
pandas version = 2.3.3
scipy version = 1.15.3
matplotlib version = 3.10.6


['BIAS_optimal.xlsx',
 'BIAS.xlsx',
 'GPR_pco2_cost.joblib',
 'GPR_pco2_rmse.joblib',
 'NSDEV.xlsx',
 'NRMSE_optimal.xlsx',
 'parameter_ranges.xlsx',
 'GPR_chl_bottle_corr.joblib',
 'GPR_bac_corr.joblib',
 'parameter_norm_optimal100_2.txt',
 'parameter_sets_optimal100_2.txt',
 'GPR_micro_corr.joblib',
 'SDEV_opt.xlsx',
 'optimisation_norm_2048.txt',
 'GPR_micro_cost.joblib',
 'GPR_chl_bottle_cost.joblib',
 'parameter_NRMSEpredicted_optimal100_2.txt',
 'apriori_parameter_ranges.xlsx',
 'GPR_fgco2_rmse.joblib',
 'parameter_norm_optimal100_3.txt',
 'CCOEF_opt.xlsx',
 'GPR_no3.joblib',
 'GPR_poc_cost.joblib',
 'GPR_sil.joblib',
 'GPR_sil_cost.joblib',
 'GPR_fgco2.joblib',
 'SDEV.xlsx',
 'GPR_pco2.joblib',
 'mcmc_wombat_mid_26params_1M_rmse_withSIL.h5',
 'CCOEF_optimal.xlsx',
 'mcmc_wombat_mid_26params_1M_rmse.h5',
 'GPR_sil_rmse.joblib',
 'GPR_fgco2_cost.joblib',
 'NSDEV_optimal.xlsx',
 'GPR_chl_bottle_rmse.joblib',
 'GPR_b1mb2.joblib',
 'NRMSE.xlsx',
 'CCOEF.xlsx',
 'GPR_nh4_corr.joblib',

### find the Sobol parameter sequence for the complete parameter set


In [3]:
df = pd.read_excel("apriori_parameter_ranges_SOTSexps.xlsx", sheet_name="parameter_ranges")
df

,WOMBAT-full,min,max,units,References,sensitivity analysis?,optimisation?,parameter number,optim number
0,alphabio_phy,1,4,(w/m2)-1 (mg Chl / mg C)-1,"MacIntyre et al., 2002",1,1,1,1
1,abioa_phy,0.000003,0.000017,/s,"Anderson et al., 2021 Nature Communications",1,1,2,2
2,bbioa_phy,1.04,1.08,none,"Anderson et al., 2021 Nature Communications",1,1,3,3
3,alphabio_dia,1,4,(w/m2)-1 (mg Chl / mg C)-1,"MacIntyre et al., 2002; Edwards et al., 2015; ...",1,0,4,3
4,alphabio_tri,0.25,2,(w/m2)-1 (mg Chl / mg C)-1,"Masotti et al., 2007 Marine Ecology Progress S...",1,0,5,3
...,...,...,...,...,...,...,...,...,...
149,bac_C2Fe,1/40e-6,1/40e-6,mol C / mol Fe,Fourquez et al. 2020,0,0,85,38
150,baclmor,-4,-1,/s,Baker & Geider 2021,1,1,86,39
151,bacqmor,0.0,0.000003,(mmol C / m3)-1 day-1,Suttle 1994,1,1,87,40
152,aox_knh4,0.45,0.45,mmol NH4 / m3,"Awata et al., 2013",0,0,87,40


### select only the parameters we are going to vary and collect information

In [5]:
param_ranges = df[df["optimisation?"]==1]

param_ranges = param_ranges.drop(["units", "References", "sensitivity analysis?", "optimisation?", "parameter number"], axis=1)
param_ranges = param_ranges.set_index("WOMBAT-full")
param_ranges



,min,max,optim number
WOMBAT-full,,,
alphabio_phy,1,4,1
abioa_phy,0.000003,0.000017,2
bbioa_phy,1.04,1.08,3
abioa_dia,0.000006,0.000017,4
bbioh,1.06,1.08,5
phykf,0.1,3,6
phymaxqc,0.01,0.08,7
phylmor,-4,-1,8
diamaxqs,0.3,1,9


In [6]:
pd.set_option("display.precision", 16)
param_ranges.loc['abioa_phy']

min             0.0000028935185185
max             0.0000173611111111
optim number                     2
Name: abioa_phy, dtype: object

### Create a normalized sobol sample

In [7]:
dim = len(param_ranges)
exps = 2048
sampler = qmc.Sobol(d=dim, scramble=True, seed=10)
sample_qmc = sampler.random(n=exps)  #here, n=256=2^8 
sample_qmc.shape


(2048, 40)

### Create the parameter sets for our sensitivity experiments

In [8]:
param_sets = np.zeros((exps,dim))

# Loop over variables correctly
for ii, var in enumerate(param_ranges.index):
    param_sets[:, ii] = (
        param_ranges.loc[var, "min"]
        + (param_ranges.loc[var, "max"] - param_ranges.loc[var, "min"])
        * sample_qmc[:, ii]
    )

names = param_ranges.index
df_exps = pd.DataFrame(param_sets, columns=names)
df_norm = pd.DataFrame(sample_qmc, columns=names)
df_norm


WOMBAT-full,alphabio_phy,abioa_phy,bbioa_phy,abioa_dia,bbioh,phykf,phymaxqc,phylmor,diamaxqs,diaVmaxs,...,dfefloor,kscav_dfe,kcoag_dfe,bsi_alpha,aoalmor,pbac_alpha,lbac_alpha,lbac_beta,baclmor,bacqmor
0,0.9870995162054896,0.8023925330489874,0.0114757269620895,0.1757747754454613,0.8299645446240902,0.0203756187111139,0.0286230696365237,0.3057573009282351,0.7182503324002028,0.0698164757341146,...,0.5271242065355182,0.4954705806449056,0.7204703763127327,0.5935327904298902,0.4458240028470755,0.5873618414625525,0.6644325936213136,0.6798762008547783,0.6020620465278625,0.5283481068909168
1,0.3413583924993873,0.4551722891628742,0.5482900263741612,0.7496405337005854,0.1669517792761326,0.7728475667536259,0.5998650537803769,0.7530542640015483,0.2724326606839895,0.8627963690087199,...,0.1826447658240795,0.7479691542685032,0.4113608328625560,0.2377206562086940,0.7839077664539218,0.2031960291787982,0.4836531616747379,0.4172046305611730,0.2994204936549067,0.3546939985826612
2,0.0944626582786441,0.5123487664386630,0.4748283578082919,0.3202799083665013,0.3155090333893895,0.6027799732983112,0.4833724685013294,0.7416576324030757,0.2099609263241291,0.5025742063298821,...,0.9150678459554911,0.0706013403832912,0.9781378852203488,0.4247273290529847,0.7192293424159288,0.4954086923971772,0.8291101735085249,0.8136528301984072,0.9480485422536731,0.8330588927492499
3,0.7031170642003417,0.2295986944809556,0.9653772767633200,0.7613868406042457,0.6797012882307172,0.3540311697870493,0.8964440021663904,0.2004919070750475,0.7807404212653637,0.3030998148024082,...,0.2595472903922200,0.8271358655765653,0.1693334709852934,0.7795476457104087,0.0100213577970862,0.8614943893626332,0.0216122148558497,0.0802343254908919,0.1414356455206871,0.0344192665070295
4,0.5002636732533574,0.7477942565456033,0.7014096165075898,0.9791394229978323,0.5122141847386956,0.1519583687186241,0.3190950080752373,0.5411454746499658,0.0041285129263997,0.8844950031489134,...,0.7845709826797247,0.2969862250611186,0.0715647116303444,0.2729823151603341,0.8815941642969847,0.0169329615309834,0.3678009305149317,0.1393595030531287,0.7762617832049727,0.2035765601322055
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2043,0.5009163711220026,0.2281865952536464,0.4658276168629527,0.5878740511834621,0.3193750753998756,0.8040389763191342,0.3016532696783543,0.5234853876754642,0.7484572120010853,0.6211026571691036,...,0.8203140925616026,0.1105496548116207,0.0009465282782912,0.3177372859790921,0.7478225920349360,0.4168705334886909,0.4575026193633676,0.9103486640378833,0.4120926950126886,0.8667233726009727
2044,0.7024261504411697,0.7490828586742282,0.2028365842998028,0.6806063102558255,0.4878244362771511,0.6936345463618636,0.9763837661594152,0.2494021579623222,0.4737995909526944,0.1913016941398382,...,0.3463143454864621,0.5154512170702219,0.2234711954370141,0.8247909201309085,0.3606210304424167,0.6968891909345984,0.1825975850224495,0.8678541658446193,0.5466747181490064,0.8796950373798609
2045,0.0938109122216702,0.0241141254082322,0.7392840245738626,0.2389500951394439,0.5084206797182560,0.4432988250628114,0.4031886830925941,0.6927323127165437,0.5436331899836659,0.9913083789870143,...,0.9398248512297869,0.2591760484501719,0.9064201209694147,0.4689940484240651,0.9018249865621328,0.0634562773630023,0.9978985534980893,0.1051331218332052,0.3650617953389883,0.2372948033735156
2046,0.3410345837473869,0.9434083607047796,0.2834979360923171,0.8251720219850540,0.9702102364972234,0.1832670802250504,0.5199252599850297,0.7979056769981980,0.9811144610866904,0.6310872891917825,...,0.2034979350864887,0.9343467606231570,0.4810014478862286,0.1579333906993270,0.5876354863867164,0.3551191473379731,0.3160156300291419,0.6266536945477128,0.8882015924900770,0.6999699482694268


### Save the parameter sets to an excel spreadsheet

In [9]:
os.chdir(wrkdir)
df_exps.to_excel('/g/data/vn19/pjb581/SOTS-Optimization/WOMBATfull/data/optimisation_sets_2048.xlsx')
df_exps.to_csv('/g/data/vn19/pjb581/SOTS-Optimization/WOMBATfull/data/optimisation_sets_2048.txt', sep=' ')
df_norm.to_excel('/g/data/vn19/pjb581/SOTS-Optimization/WOMBATfull/data/optimisation_norm_2048.xlsx')
df_norm.to_csv('/g/data/vn19/pjb581/SOTS-Optimization/WOMBATfull/data/optimisation_norm_2048.txt', sep=' ')


In [10]:
os.getcwd()

'/g/data/vn19/pjb581/SOTS-Optimization/WOMBATfull/data'

In [11]:
df_exps

WOMBAT-full,alphabio_phy,abioa_phy,bbioa_phy,abioa_dia,bbioh,phykf,phymaxqc,phylmor,diamaxqs,diaVmaxs,...,dfefloor,kscav_dfe,kcoag_dfe,bsi_alpha,aoalmor,pbac_alpha,lbac_alpha,lbac_beta,baclmor,bacqmor
0,3.9612985486164689,0.0000145022067860,1.0404590290784836,0.0000078214673084,1.0765992908924820,0.1590892942622304,0.0120036148745567,-3.0827280972152948,0.8027752326801418,0.0000008078170274,...,0.0268290861202404,-2.0181176774203777,-5.3976481184363365,-8.5161680239252746,-2.6625279914587736,0.5873618414625525,0.6312109639402479,0.6798762008547783,-2.1938138604164124,0.0000015833743710
1,2.0240751774981618,0.0000094787657576,1.0619316010549664,0.0000144634321030,1.0633390355855228,2.3412579435855152,0.0519905537646264,-1.7408372079953551,0.4907028624787926,0.0000060392816011,...,0.0099495935253799,-1.0081233829259872,-6.9431958356872201,-9.4056983594782650,-1.6482767006382346,0.2031960291787982,0.4594705035910010,0.4172046305611730,-3.1017385190352798,0.0000011010018479
2,1.2833879748359323,0.0000103059717367,1.0589931343123318,0.0000094939804209,1.0663101806677879,1.8480619225651027,0.0438360727950931,-1.7750271027907729,0.4469726484268903,0.0000036628159445,...,0.0458383244518191,-3.7175946384668350,-4.1093105738982558,-8.9381816773675382,-1.8423119727522135,0.4954086923971772,0.7876546648330986,0.8136528301984072,-1.1558543732389808,0.0000024297932206
3,3.1093511926010251,0.0000062152588901,1.0786150910705328,0.0000145993847292,1.0735940257646144,1.1266903923824430,0.0727510801516473,-3.3985242787748575,0.8465182948857546,0.0000023468390560,...,0.0137178172292188,-0.6914565376937389,-8.1533326450735331,-8.0511308857239783,-3.9699359266087413,0.8614943893626332,0.0205316041130573,0.0802343254908919,-3.5756930634379387,0.0000002113498144
4,2.5007910197600722,0.0000137123011653,1.0680563846603037,0.0000171196692477,1.0702442836947739,0.5406792692840099,0.0323366505652666,-2.3765635760501027,0.3028899590484798,0.0000061824323124,...,0.0394439781513065,-2.8120550997555256,-8.6421764418482780,-9.3175442120991647,-1.3552175071090460,0.0169329615309834,0.3494108839891851,0.1393595030531287,-1.6712146503850818,0.0000006812311856
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2043,2.5027491133660078,0.0000061948292137,1.0586331046745181,0.0000125911348517,1.0663875015079975,2.4317130313254891,0.0311157288774848,-2.4295438369736075,0.8239200484007596,0.0000044447744744,...,0.0411953905355185,-3.5578013807535172,-8.9952673586085439,-9.2056567850522697,-1.7565322238951921,0.4168705334886909,0.4346274883951992,0.9103486640378833,-2.7637219149619341,0.0000025233056646
2044,3.1072784513235092,0.0000137309441359,1.0481134633719922,0.0000136644248872,1.0697564887255431,2.1115401844494044,0.0783468636311591,-3.2517935261130333,0.6316597136668860,0.0000016092820100,...,0.0179694029288366,-1.9381951317191124,-7.8826440228149295,-7.9380226996727288,-2.9181369086727500,0.6968891909345984,0.1734677057713270,0.8678541658446193,-2.3599758455529809,0.0000025593380668
2045,1.2814327366650105,0.0000032423918607,1.0695713609829545,0.0000085526631382,1.0701684135943652,1.3855665926821530,0.0382232078164816,-1.9218030618503690,0.6805432329885661,0.0000068871038892,...,0.0470514177102596,-2.9632958061993122,-4.4678993951529264,-8.8275148789398372,-1.2945250403136015,0.0634562773630023,0.9480036258231848,0.1051331218332052,-2.9048146139830351,0.0000007748929723
2046,2.0231037512421608,0.0000165423663296,1.0513399174436928,0.0000153376391433,1.0794042047299446,0.6314745326526463,0.0463947681989521,-1.6062829690054059,0.9867801227606832,0.0000045106453106,...,0.0109713988192379,-0.2626129575073719,-6.5949927605688572,-9.6051665232516825,-2.2370935408398509,0.3551191473379731,0.3002148485276848,0.6266536945477128,-1.3353952225297689,0.0000020601017082
